# Natural Language Processing Coursework

## Setup

In [1]:


import os
if not os.path.exists("NLPLabs-2024"):
    !git clone -q https://github.com/CRLala/NLPLabs-2024.git


if not os.path.exists("dontpatronizeme"):
    !git clone -q https://github.com/Perez-AlmendrosC/dontpatronizeme.git

    
pcl_tsv_path = "NLPLabs-2024/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
train_split_path = "dontpatronizeme/semeval-2022/practice splits/train_semeval_parids-labels.csv"
dev_split_path   = "dontpatronizeme/semeval-2022/practice splits/dev_semeval_parids-labels.csv"


In [2]:
import os
from pathlib import Path
import re
import csv
import numpy as np
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

In [3]:
from huggingface_hub import login
import os
# Option 1: token from environment variable (recommended)
token = os.getenv("HF_TOKEN")
if token is not None:
    login(token=token)
else:
    # Interactive login (will prompt in notebook / terminal)
    login()

In [4]:
rows=[]
with open(pcl_tsv_path) as f:
    for line in f.readlines()[4:]:
        par_id=line.strip().split('\t')[0]
        art_id = line.strip().split('\t')[1]
        keyword=line.strip().split('\t')[2]
        country=line.strip().split('\t')[3]
        t=line.strip().split('\t')[4]#.lower()
        l=line.strip().split('\t')[-1]
        if l=='0' or l=='1':
            lbin=0
        else:
            lbin=1
        rows.append(
            {'par_id':int(par_id),
            'doc_id':art_id,
            'keyword':keyword,
            'country':country,
            'text':t, 
            'label':lbin, 
            'orig_label':int(l)
            }
        )
df=pd.DataFrame(rows, columns=['par_id', 'doc_id', 'keyword', 'country', 'text', 'label', 'orig_label'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10469 entries, 0 to 10468
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   par_id      10469 non-null  int64
 1   doc_id      10469 non-null  str  
 2   keyword     10469 non-null  str  
 3   country     10469 non-null  str  
 4   text        10469 non-null  str  
 5   label       10469 non-null  int64
 6   orig_label  10469 non-null  int64
dtypes: int64(3), str(4)
memory usage: 3.4 MB


In [5]:
import ast
import numpy as np
import pandas as pd


train_split = pd.read_csv(train_split_path)
dev_split   = pd.read_csv(dev_split_path)

# Parse 7-dim multi-hot label vectors
def parse_label_vec(x):
    v = ast.literal_eval(x) if isinstance(x, str) else x
    if not isinstance(v, list) or len(v) != 7:
        raise ValueError(f"Expected 7-dim list, got: {x}")
    # float for BCEWithLogitsLoss in multi-label training
    return [float(t) for t in v]

train_split["par_id"] = train_split["par_id"].astype(int)
dev_split["par_id"]   = dev_split["par_id"].astype(int)
train_split["label_vec"] = train_split["label"].apply(parse_label_vec)
dev_split["label_vec"]   = dev_split["label"].apply(parse_label_vec)

# Build train/dev dataframes based on par_id
train_ids = set(train_split["par_id"].tolist())
dev_ids   = set(dev_split["par_id"].tolist())

df_train = df[df["par_id"].isin(train_ids)].copy().reset_index(drop=True)
df_dev   = df[df["par_id"].isin(dev_ids)].copy().reset_index(drop=True)

# Attach label_vec
train_map = dict(zip(train_split["par_id"], train_split["label_vec"]))
dev_map   = dict(zip(dev_split["par_id"], dev_split["label_vec"]))

df_train["label_vec"] = df_train["par_id"].map(train_map)
df_dev["label_vec"]   = df_dev["par_id"].map(dev_map)

# Sanity checks
assert df_train["label_vec"].notna().all(), "Some train rows missing label_vec (par_id mismatch)"
assert df_dev["label_vec"].notna().all(), "Some dev rows missing label_vec (par_id mismatch)"
assert set(df_train["label"].unique()).issubset({0, 1}), "Binary label column must be 0/1"
assert set(df_dev["label"].unique()).issubset({0, 1}), "Binary label column must be 0/1"

print("Train:", df_train.shape, "pos_rate:", df_train["label"].mean())
print("Dev:  ", df_dev.shape,   "pos_rate:", df_dev["label"].mean())

Train: (8375, 8) pos_rate: 0.09480597014925374
Dev:   (2094, 8) pos_rate: 0.09503342884431709


In [6]:
df_train.tail()

,par_id,doc_id,keyword,country,text,label,orig_label,label_vec
8370,10424,@@4665292,women,jm,""""""" I do n't believe in abortion , I think it ...",1,3,"[1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]"
8371,10445,@@3923193,refugee,gb,More than 150 volunteers spent the night in ' ...,1,3,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
8372,10454,@@22338535,vulnerable,ie,""""""" We are challenged , I suggest , to turn th...",1,4,"[1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0]"
8373,10467,@@20282330,in-need,ng,""""""" She has one huge platform , and informatio...",1,3,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
8374,10469,@@16779383,homeless,ie,""""""" Guinness World Record of 540lbs of 7-layer...",1,3,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


# 4. Model

In [7]:
import os
import random
import numpy as np
import torch
from torch import nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


4.1 Tokenization

In [8]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256  # Justification: EDA shows p99 ~160 tokens, with neglible occurences > 256 across both classes. Boosts eff

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )

# 7 dim
train_ds_task2 = Dataset.from_pandas(df_train[["text", "label_vec"]].rename(columns={"label_vec": "labels"}), preserve_index=False)
dev_ds_task2   = Dataset.from_pandas(df_dev[["text", "label_vec"]].rename(columns={"label_vec": "labels"}), preserve_index=False)

# binary
train_ds_task1 = Dataset.from_pandas(df_train[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
dev_ds_task1   = Dataset.from_pandas(df_dev[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)

# Tokenize
train_ds_task2 = train_ds_task2.map(tokenize_fn, batched=True, remove_columns=["text"])
dev_ds_task2   = dev_ds_task2.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds_task1 = train_ds_task1.map(tokenize_fn, batched=True, remove_columns=["text"])
dev_ds_task1   = dev_ds_task1.map(tokenize_fn, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if device == "cuda" else None
)

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics_task2(eval_pred):
    logits, labels = eval_pred
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    y = labels.astype(int)
    ### Micro F1 stable under label imbalance and standard for multi-;abel
    tp = (preds & y).sum()
    fp = (preds & (1 - y)).sum()
    fn = ((1 - preds) & y).sum()
    prec = tp / (tp + fp + 1e-12)
    rec  = tp / (tp + fn + 1e-12)
    f1_micro = 2 * prec * rec / (prec + rec + 1e-12)

    return {"f1_micro": float(f1_micro), "precision_micro": float(prec), "recall_micro": float(rec)}

# Justification: We use Task2 only to shape the encoder toward PCL-related cues,
# problem_type ensures BCEWithLogitsLoss, appropriate for multi-label targets
model_task2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=7,
    problem_type="multi_label_classification"
).to(device)

args_task2 = TrainingArguments(
    output_dir="runs/task2_intermediate",
    seed=SEED,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,

    # transformers>=5 uses eval_strategy (not evaluation_strategy)
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
    save_total_limit=2,
)

trainer_task2 = Trainer(
    model=model_task2,
    args=args_task2,
    train_dataset=train_ds_task2,
    eval_dataset=dev_ds_task2,
    data_collator=data_collator,
    compute_metrics=compute_metrics_task2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer_task2.train()

task2_best_dir = "checkpoints/task2_best"
os.makedirs(task2_best_dir, exist_ok=True)
trainer_task2.save_model(task2_best_dir)
tokenizer.save_pretrained(task2_best_dir)
print("Saved Task2 checkpoint:", task2_best_dir)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Micro,Precision Micro,Recall Micro
1,0.100043,0.100430,0.197719,0.658228,0.116331


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Compute class weights from true distribution (no negative downsampling)
neg = int((df_train["label"] == 0).sum())
pos = int((df_train["label"] == 1).sum())
w_pos = neg / max(pos, 1)


class_weights = torch.tensor([1.0, float(w_pos)], dtype=torch.float32, device=device)
print(f"neg={neg} pos={pos} class_weights={class_weights.tolist()}")

class WeightedCETrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics_task1(eval_pred):
    logits, labels = eval_pred
    probs_pos = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    auprc = average_precision_score(labels, probs_pos)

    thresholds = np.linspace(0.0, 1.0, 201)
    best_f1, best_t = -1.0, 0.5
    for t in thresholds:
        preds = (probs_pos >= t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    preds05 = (probs_pos >= 0.5).astype(int)
    return {
        "auprc": float(auprc),
        "f1_best": float(best_f1),
        "best_threshold_coarse": float(best_t),
        "precision@0.5": float(precision_score(labels, preds05, zero_division=0)),
        "recall@0.5": float(recall_score(labels, preds05, zero_division=0)),
        "f1@0.5": float(f1_score(labels, preds05, zero_division=0)),
    }

model_task1 = AutoModelForSequenceClassification.from_pretrained(
    task2_best_dir,
    num_labels=2,
    ignore_mismatched_sizes=True
).to(device)

args_task1 = TrainingArguments(
    output_dir="runs/task1_binary",
    seed=SEED,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_best",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
    save_total_limit=2,
)

trainer_task1 = WeightedCETrainer(
    class_weights=class_weights,
    model=model_task1,
    args=args_task1,
    train_dataset=train_ds_task1,
    eval_dataset=dev_ds_task1,
    data_collator=data_collator,
    compute_metrics=compute_metrics_task1,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer_task1.train()

task1_best_dir = "checkpoints/task1_best"
os.makedirs(task1_best_dir, exist_ok=True)
trainer_task1.save_model(task1_best_dir)
tokenizer.save_pretrained(task1_best_dir)
print("Saved Task1 checkpoint:", task1_best_dir)

In [ ]:
pred_out = trainer_task1.predict(dev_ds_task1)
dev_logits = pred_out.predictions
dev_labels = pred_out.label_ids

dev_probs_pos = torch.softmax(torch.tensor(dev_logits), dim=-1)[:, 1].numpy()

# Fine grid for final threshold selection (metric-aligned)
thresholds = np.linspace(0.0, 1.0, 2001)
best_f1, best_t = -1.0, 0.5
for t in thresholds:
    preds = (dev_probs_pos >= t).astype(int)
    f1 = f1_score(dev_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

dev_preds = (dev_probs_pos >= best_t).astype(int)
p = precision_score(dev_labels, dev_preds, zero_division=0)
r = recall_score(dev_labels, dev_preds, zero_division=0)
auprc = average_precision_score(dev_labels, dev_probs_pos)

print(f"Dev tuned threshold: {best_t:.4f}")
print(f"Dev Precision: {p:.4f}  Recall: {r:.4f}  F1: {best_f1:.4f}  AUPRC: {auprc:.4f}")

with open("dev.txt", "w", encoding="utf-8") as f:
    for y in dev_preds:
        f.write(f"{int(y)}\n")
print("Wrote dev.txt")

In [ ]:
BESTMODEL_DIR = "BestModel"
os.makedirs(BESTMODEL_DIR, exist_ok=True)

final_model = AutoModelForSequenceClassification.from_pretrained(task1_best_dir)
final_model.save_pretrained(BESTMODEL_DIR)
tokenizer.save_pretrained(BESTMODEL_DIR)

with open(os.path.join(BESTMODEL_DIR, "threshold.txt"), "w", encoding="utf-8") as f:
    f.write(str(best_t) + "\n")

with open(os.path.join(BESTMODEL_DIR, "training_notes.txt"), "w", encoding="utf-8") as f:
    f.write(f"MODEL={MODEL_NAME}\n")
    f.write(f"MAX_LEN={MAX_LEN}\n")
    f.write(f"SEED={SEED}\n")
    f.write(f"neg={neg} pos={pos} class_weights={class_weights.tolist()}\n")
    f.write("Approach: Task2 intermediate fine-tuning -> Task1 weighted fine-tuning + dev threshold tuning.\n")

print("Saved BestModel/ (model + tokenizer + threshold).")